<a href="https://colab.research.google.com/github/zeyneptass/vector-databases-and-rag-fundamentals/blob/main/hybrid_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pinecone pinecone-text sentence-transformers datasets tqdm pandas

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 14.6 MB/s eta 0:00:00


In [ ]:
import warnings
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from pinecone_text.sparse import BM25Encoder
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

In [ ]:
from google.colab import userdata

PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')
pc = Pinecone(api_key=PINECONE_API_KEY)

print("Pinecone istemcisi hazır.")

Pinecone istemcisi hazır.


In [ ]:
# Hugging Face'ten moda ürün görselleri veri setini yükle
dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:5000]")

print(f"Toplam {len(dataset)} ürün yüklendi.")
print("Örnek:", dataset[0]['productDisplayName'])

README.md:   0%|          | 0.00/867 [00:00<?, ?B/s]

data/train-00000-of-00002-6cff4c59f91661(…): reconstructing file:   0%|          |  0.00B /  136MB            

data/train-00000-of-00002-6cff4c59f91661(…): downloading bytes:           |  0.00B            

data/train-00001-of-00002-bb459e5ac5f01e(…): reconstructing file:   0%|          |  0.00B /  135MB            

data/train-00001-of-00002-bb459e5ac5f01e(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/44072 [00:00<?, ? examples/s]

Toplam 5000 ürün yüklendi.
Örnek: Turtle Check Men Navy Blue Shirt


In [ ]:
# Metadata'yı Pandas DataFrame'e çevir
metadata = dataset['train'].to_pandas() if 'train' in dataset else dataset.to_pandas()

# İlk 5 satıra bak
print(metadata.head())
print(f"\nSütunlar: {metadata.columns.tolist()}")

      id gender masterCategory subCategory  articleType baseColour  season  \
0  15970    Men        Apparel     Topwear       Shirts  Navy Blue    Fall   
1  39386    Men        Apparel  Bottomwear        Jeans       Blue  Summer   
2  59263  Women    Accessories     Watches      Watches     Silver  Winter   
3  21379    Men        Apparel  Bottomwear  Track Pants      Black    Fall   
4  53759    Men        Apparel     Topwear      Tshirts       Grey  Summer   

     year   usage                             productDisplayName  \
0  2011.0  Casual               Turtle Check Men Navy Blue Shirt   
1  2012.0  Casual             Peter England Men Party Blue Jeans   
2  2016.0  Casual                       Titan Women Silver Watch   
3  2011.0  Casual  Manchester United Men Solid Black Track Pants   
4  2012.0  Casual                          Puma Men Grey T-shirt   

                                               image  
0  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...  
1  {'bytes':

In [ ]:
# BM25 encoder'ı oluştur
bm25_encoder = BM25Encoder()

# Ürün açıklamalarını kullanarak encoder'ı eğit
# Bu, tüm kelime dağarcığını ve frekansları öğrenir
product_names = metadata['productDisplayName'].tolist()
bm25_encoder.fit(product_names)

print("BM25 encoder eğitildi.")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


  0%|          | 0/5000 [00:00<?, ?it/s]

BM25 encoder eğitildi.


In [ ]:
# Bir ürün adını sparse vektöre çevir
test_sparse = bm25_encoder.encode_documents("Turtle Check Men Navy Blue Shirt")
print("Sparse vektör örneği:")
print(f"  Indices: {test_sparse['indices'][:10]}...")  # İlk 10 index
print(f"  Values: {test_sparse['values'][:10]}...")    # İlk 10 değer
print(f"  Toplam dolu boyut: {len(test_sparse['indices'])}")

Sparse vektör örneği:
  Indices: [23789636, 1830646559, 632192512, 931643408, 3905155331, 3828986392]...
  Values: [0.44563479821633906, 0.44563479821633906, 0.44563479821633906, 0.44563479821633906, 0.44563479821633906, 0.44563479821633906]...
  Toplam dolu boyut: 6


In [ ]:
# CLIP modelini yükle (görsel-metin modeli)
# Bu model, görselleri ve metinleri aynı vektör uzayına çevirir
clip_model = SentenceTransformer('clip-ViT-B-32')

print(f"CLIP model boyutu: {clip_model.get_sentence_embedding_dimension()}")

modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/1.91k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.03k [00:00<?, ?B/s]

0_CLIPModel/model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

...

tokenizer_config.json:   0%|          | 0.00/604 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/961k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIP model boyutu: 512


In [ ]:
# Bir metni dense vektöre çevir
test_dense = clip_model.encode("Turtle Check Men Navy Blue Shirt")
print(f"Dense vektör boyutu: {len(test_dense)}")
print(f"İlk 5 değer: {test_dense[:5]}")

Dense vektör boyutu: 512
İlk 5 değer: [ 0.34759915  0.18605334 -0.03437042 -0.08769464  0.21245079]


In [ ]:
index_name = "hybrid-search-demo"

# Varsa eski dizini sil
if index_name in pc.list_indexes().names():
    print(f"'{index_name}' siliniyor...")
    pc.delete_index(index_name)

# Yeni dizin oluştur (dot product metriği ile)
pc.create_index(
    name=index_name,
    dimension=512,  # CLIP modelinin boyutu
    metric="dotproduct",  # Hibrit arama için dot product
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)

print(f"'{index_name}' dizini oluşturuldu.")

'hybrid-search-demo' dizini oluşturuldu.


In [ ]:
import time

while not pc.describe_index(index_name).status.ready:
    print("Dizin hazırlanıyor...")
    time.sleep(2)

print("Dizin hazır!")

# Dizine bağlan
index = pc.Index(index_name)
print(index)

Dizin hazır!
Index(host='https://hybrid-search-demo-h0jgoc3.svc.aped-4627-b74a.pinecone.io')


In [ ]:
batch_size = 100
upserts = []

for i in tqdm(range(0, len(metadata), batch_size)):
    # Bu partideki verileri al
    batch = metadata.iloc[i:i + batch_size]

    # Ürün adlarını al
    product_names = batch['productDisplayName'].tolist()
    ids = [f"product_{i + j}" for j in range(len(batch))]

    # Dense vektörleri oluştur (CLIP ile)
    dense_vectors = clip_model.encode(product_names).tolist()

    # Pinecone formatına hazırla
    for j in range(len(batch)):
        # Her ürün adı için sparse vektörü ayrı ayrı oluştur
        sparse_vector = bm25_encoder.encode_documents(product_names[j])

        upserts.append({
            "id": ids[j],
            "values": dense_vectors[j],           # Dense vektör
            "sparse_values": {                     # Sparse vektör
                "indices": sparse_vector['indices'],
                "values": sparse_vector['values']
            },
            "metadata": {
                "productDisplayName": product_names[j],
                "gender": batch.iloc[j]['gender'],
                "articleType": batch.iloc[j]['articleType'],
                "baseColour": batch.iloc[j]['baseColour']
            }
        })

    # 200'e ulaşınca yükle
    if len(upserts) >= 200:
        index.upsert(vectors=upserts)
        upserts = []

# Kalanları yükle
if upserts:
    index.upsert(vectors=upserts)

print("Tüm veriler yüklendi!")

  0%|          | 0/50 [00:00<?, ?it/s]

Tüm veriler yüklendi!


In [ ]:
stats = index.describe_index_stats()
print(f"Toplam vektör sayısı: {stats['total_vector_count']}")

Toplam vektör sayısı: 5000


In [ ]:
def hybrid_search(query_text, alpha=0.5, top_k=5):
    """
    Hibrit arama yapar: sparse + dense vektörleri birlikte kullanır.

    alpha: 0 = tamamen sparse, 1 = tamamen dense
    """
    # Sorguyu hem sparse hem dense vektöre çevir
    query_sparse = bm25_encoder.encode_queries(query_text)
    query_dense = clip_model.encode(query_text).tolist()

    # Alpha'ya göre ölçeklendir
    # Dense'i alpha ile, sparse'i (1-alpha) ile çarp
    scaled_dense = [v * alpha for v in query_dense]
    scaled_sparse = {
        "indices": query_sparse['indices'],
        "values": [v * (1 - alpha) for v in query_sparse['values']]
    }

    # Pinecone'da ara
    results = index.query(
        vector=scaled_dense,
        sparse_vector=scaled_sparse,
        top_k=top_k,
        include_metadata=True
    )

    # Sonuçları yazdır
    print(f"\nSorgu: '{query_text}' (alpha={alpha})")
    print("-" * 80)
    for match in results['matches']:
        score = match['score']
        name = match['metadata']['productDisplayName']
        gender = match['metadata']['gender']
        article = match['metadata']['articleType']
        print(f"Skor: {score:.4f} | {name} | {gender} | {article}")
    print("-" * 80)

    return results

In [ ]:
# Test 1: Dengeli arama (alpha=0.5)
hybrid_search("dark blue French connection jeans for men", alpha=0.5)

# Test 2: Tamamen dense (alpha=1.0)
hybrid_search("dark blue French connection jeans for men", alpha=1.0)

# Test 3: Tamamen sparse (alpha=0.0)
hybrid_search("dark blue French connection jeans for men", alpha=0.0)

# Test 4: Dense ağırlıklı (alpha=0.8)
hybrid_search("dark blue French connection jeans for men", alpha=0.8)

# Test 5: Sparse ağırlıklı (alpha=0.2)
hybrid_search("dark blue French connection jeans for men", alpha=0.2)


Sorgu: 'dark blue French connection jeans for men' (alpha=0.5)
--------------------------------------------------------------------------------
Skor: 34.9431 | French Connection Men Blue Jeans | Men | Jeans
Skor: 34.9431 | French Connection Men Blue Jeans | Men | Jeans
Skor: 31.3047 | 18+ Men Thrill Deo | Men | Deodorant
Skor: 31.1470 | Lee Men Tino Blue Jeans | Men | Jeans
Skor: 30.5848 | John Players Men Blue Jeans | Men | Jeans
--------------------------------------------------------------------------------

Sorgu: 'dark blue French connection jeans for men' (alpha=1.0)
--------------------------------------------------------------------------------
Skor: 69.5327 | French Connection Men Blue Jeans | Men | Jeans
Skor: 69.5327 | French Connection Men Blue Jeans | Men | Jeans
Skor: 62.5924 | 18+ Men Thrill Deo | Men | Deodorant
Skor: 62.1464 | Lee Men Tino Blue Jeans | Men | Jeans
Skor: 61.0219 | John Players Men Blue Jeans | Men | Jeans
-----------------------------------------------

QueryResponse(matches=[ScoredVector(id='product_1662', score=14.1893177, values=[], metadata={'articleType': 'Jeans', 'baseColour': 'Blue', 'gender': 'Men', 'productDisplayName': 'French Connection Men Blue Jeans'}), ScoredVector(id='product_4653', score=14.1892929, values=[], metadata={'articleType': 'Jeans', 'baseColour': 'Blue', 'gender': 'Men', 'productDisplayName': 'French Connection Men Blue Jeans'}), ScoredVector(id='product_1871', score=12.5474739, values=[], metadata={'articleType': 'Jeans', 'baseColour': 'Blue', 'gender': 'Men', 'productDisplayName': 'Lee Men Tino Blue Jeans'}), ScoredVector(id='product_1665', score=12.532073, values=[], metadata={'articleType': 'Deodorant', 'baseColour': 'Blue', 'gender': 'Men', 'productDisplayName': '18+ Men Thrill Deo'}), ScoredVector(id='product_1636', score=12.3224983, values=[], metadata={'articleType': 'Jeans', 'baseColour': 'Blue', 'gender': 'Men', 'productDisplayName': 'John Players Men Blue Jeans'})], namespace='', usage=Usage(read_

In [ ]:
def display_results(results):
    """
    Arama sonuçlarını görsel olarak gösterir.
    """
    for match in results['matches']:
        name = match['metadata']['productDisplayName']
        score = match['score']
        print(f"Skor: {score:.4f} | {name}")